In [1]:
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import datetime
from datetime import datetime
import os
from tqdm import tqdm
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
import plotly.express as px
import plotly.graph_objects as go
from pptx.dml.color import RGBColor

In [2]:
# データ準備
df = pd.DataFrame([
    dict(Task="2HTS", Start='2025-10-10', Finish='2025-12-01', Resource='大橋さん', Progress=0.70),
    dict(Task="MD_福地さん案件", Start='2025-12-05', Finish='2025-12-26', Resource='福地さん', Progress=0.70),
    # dict(Task="", Start='2025-11-07', Finish='2025-12-12', Resource='1UHSL XCアップデート', Progress=0.90),
    # dict(Task="MD_岡村さん", Start='2025-12-03', Finish='2025-12-12', Resource='2UHSL XCアップデート', Progress=0.90),
    dict(Task="防錆刷新プロジェクト", Start='2025-11-20', Finish='2026-06-30', Resource='松本さん', Progress=0.18),
    # dict(Task="8SL", Start='2025-12-8', Finish='2025-11-28', Resource='人事関連', Progress=1.0),
    #dict(Task="データ確認", Start='2026-1-13', Finish='2026-1-30', Resource='平山さん', Progress=0.3),
    dict(Task="python共有会", Start='2025-10-10', Finish='2025-12-20', Resource='啓発関連', Progress=0.50),
])

df["Start"] = pd.to_datetime(df["Start"])
df["Finish"] = pd.to_datetime(df["Finish"])
fig = px.timeline(df, x_start="Start", x_end="Finish", y="Task", color="Resource")
fig.update_yaxes(autorange="reversed")

# 今日の日付(日時)
today = datetime.now()

# 縦線をshapeとして追加
fig.add_shape(
    type="line",
    x0=today,
    x1=today,
    y0=-0.5,           # y軸はカテゴリなので少し余裕をもって指定
    y1=len(df["Task"])-0.5,
    line=dict(color="red", width=2, dash="dash"),
)

# 進捗率テキストをバーの中央に表示
for _, row in df.iterrows():
    midpoint = row["Start"] + (row["Finish"] - row["Start"]) / 2
    fig.add_trace(go.Scatter(
        x=[midpoint],
        y=[row["Task"]],
        mode="text",
        text=[f"{int(row['Progress'] * 100)}%"],
        textposition="middle center",
        showlegend=False,
        textfont=dict(color="white", size=12, family="Arial Black"),
    ))
fig.write_image("../data/gantt_chart.png")
fig.show()

In [3]:
df_NEXT_TASK = pd.DataFrame([
    dict(Task="2HTS", NextTask='馬場崎さんNASの復興待ち', 備考 = '復興にめどがついてない状態' ),
    #dict(Task="MD_岡村さん", NextTask='XCゲートの平山さんに見せて完了とした。現場へ打診して本運用に進める', 備考= '-' ),
    dict(Task="MD_福地さん", NextTask='下工程のデータ連携完了', 備考 = '' ),
    dict(Task="防錆刷新プロジェクト", NextTask='進捗打ち合わせの聴講', 備考 = '-'),
    dict(Task="データ確認", NextTask='XCゲートにデータはあるが、タブロウにはデータが上がっていない件の確認', 備考 = 'データの流れ及び原理がわかっていない。'),
    # dict(Task="Python共有会", NextTask='次の予定を考える', 備考 = "-"),
])
df_NEXT_TASK

,Task,NextTask,備考
0,2HTS,馬場崎さんNASの復興待ち,復興にめどがついてない状態
1,MD_福地さん,下工程のデータ連携完了,
2,防錆刷新プロジェクト,進捗打ち合わせの聴講,-
3,データ確認,XCゲートにデータはあるが、タブロウにはデータが上がっていない件の確認,データの流れ及び原理がわかっていない。


In [4]:
# 詳細情報の入力
print('タスク一覧取得')
print('-------------------------')


def read_files_in_folder(folder_path):
    for filename in tqdm(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path):
            print(f'{filename}')

if __name__ == "__main__":
    folder = '../data/詳細タスク/'
    read_files_in_folder(folder)

タスク一覧取得
-------------------------


100%|████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 1138.65it/s]

1NFL.xlsx
1UHSL.xlsx
2HS.xlsx
2UHSL.xlsx
3PASS.xlsx
8SL.xlsx
MD_ポリッシュ日常点検’.xlsx
MD_福地さん.xlsx
NFSL.xlsx
その他.xlsx
エネルギー管理.xlsx
クラウド.xlsx
データ系.xlsx
ナルプロ.xlsx
人事.xlsx
自部署データ関連.xlsx


In [5]:
def read_and_filter_files(folder_path):
    today_str = datetime.today().strftime('%Y-%m-%d')
    df_list = []

    for filename in tqdm(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path):
            if filename.endswith('.xlsx') or filename.endswith('.xls'):
                xls = pd.ExcelFile(file_path)
                for sheet_name in xls.sheet_names:
                    df = pd.read_excel(xls, sheet_name=sheet_name)
                    if '報告日' in df.columns:
                        df_filtered = df[df['報告日'].astype(str).str.contains(today_str)]
                        if not df_filtered.empty:
                            df_filtered['業務カテゴリ'] = sheet_name
                            df_list.append(df_filtered)
            elif filename.endswith('.csv'):
                df = pd.read_csv(file_path)
                if '報告日' in df.columns:
                    df_filtered = df[df['報告日'].astype(str).str.contains(today_str)]
                    if not df_filtered.empty:
                        category_name = os.path.splitext(filename)[0]
                        df_filtered['業務カテゴリ'] = category_name
                        df_list.append(df_filtered)
            else:
                print(f"{filename} はサポートしていない形式です。")

    if df_list:
        merged_df = pd.concat(df_list, axis=0, ignore_index=True)
        # ここで業務カテゴリでソートする
        merged_df = merged_df.sort_values(by='業務カテゴリ').reset_index(drop=True)
        return merged_df
    else:
        print("該当データがありません。")
        return pd.DataFrame()

if __name__ == "__main__":
    folder = '../data/詳細タスク/'
    merged_data = read_and_filter_files(folder)
    if not merged_data.empty:
        merged_data
merged_data.to_csv('../temp/merged_Data.csv',encoding='cp932')

 50%|█████████████████████████████████████████▌                                         | 8/16 [00:03<00:02,  3.86it/s]C:\Users\c0100106850\AppData\Local\Temp\ipykernel_17780\1481570905.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

100%|██████████████████████████████████████████████████████████████████████████████████| 16/16 [00:03<00:00,  4.53it/s]


In [6]:
# （省略）ガントチャート作成コードはここに
import datetime
# 画像保存
img_path = "gantt_chart.png"
fig.write_image(img_path)

# PowerPoint作成
prs = Presentation()
slide = prs.slides.add_slide(prs.slide_layouts[5])  # 白紙レイアウト

shapes_to_delete = [shape for shape in slide.shapes]
for shape in shapes_to_delete:
    sp = shape._element
    sp.getparent().remove(sp)


# タイトルを追加
title_shape = slide.shapes.add_textbox(Inches(1), Inches(0.3), Inches(8), Inches(1))
title_tf = title_shape.text_frame
title_tf.text = "プロジェクト進捗ガントチャート"
title_tf.paragraphs[0].font.size = Pt(32)
title_tf.paragraphs[0].font.bold = True
title_tf.paragraphs[0].alignment = PP_ALIGN.CENTER

# 説明テキストを追加
desc_shape = slide.shapes.add_textbox(Inches(1), Inches(1.0), Inches(8), Inches(1))
desc_tf = desc_shape.text_frame
desc_tf.text = "主要タスクの進捗状況"
desc_tf.paragraphs[0].font.size = Pt(18)
desc_tf.paragraphs[0].alignment = PP_ALIGN.LEFT

# ガントチャート画像を挿入
left = Inches(1)
top = Inches(1.5)
slide.shapes.add_picture(img_path, left, top, width=Inches(8))




# 2ページ目スライド作成（白紙レイアウト）
slide = prs.slides.add_slide(prs.slide_layouts[5])

# すべてのシェイプを削除
shapes_to_delete = [shape for shape in slide.shapes]
for shape in shapes_to_delete:
    sp = shape._element
    sp.getparent().remove(sp)

# タイトルを追加
title_shape = slide.shapes.add_textbox(Inches(1), Inches(0.3), Inches(8), Inches(1))
title_tf = title_shape.text_frame
title_tf.text = "プロジェクトの状況の共有"
p = title_tf.paragraphs[0]
p.font.size = Pt(24)
p.font.bold = True
p.alignment = PP_ALIGN.CENTER


rows, cols = df_NEXT_TASK.shape
# ヘッダー行を1行プラス
table_shape = slide.shapes.add_table(rows + 1, cols, Inches(0.5), Inches(1.5), Inches(9), Inches(0.8 + 0.3 * rows))
table = table_shape.table

# ヘッダーの書き込み
for col_idx, col_name in enumerate(df_NEXT_TASK.columns):
    cell = table.cell(0, col_idx)
    cell.text = str(col_name)
    para = cell.text_frame.paragraphs[0]
    para.font.bold = True
    para.font.size = Pt(12)
    cell.fill.solid()
    cell.fill.fore_color.rgb = RGBColor(200, 200, 200)  # 薄グレー背景

# データ書き込み
for row_idx in range(rows):
    for col_idx in range(cols):
        cell = table.cell(row_idx + 1, col_idx)
        cell.text = str(df_NEXT_TASK.iat[row_idx, col_idx])
        para = cell.text_frame.paragraphs[0]
        para.font.size = Pt(11)

if merged_data is not None and not merged_data.empty:
    grouped = merged_data.groupby('業務カテゴリ')

    for category, df_cat in tqdm(grouped):
        # 「No.」カラムを整数型に変換（できない値は欠損に）
        if 'No.' in df_cat.columns:
            df_cat['No.'] = pd.to_numeric(df_cat['No.'], errors='coerce')

        # 「No.」をキーに重複削除（最初の出現以外は削除）
        if 'No.' in df_cat.columns:
            df_cat = df_cat.drop_duplicates(subset=['No.'], keep='first')

        # 「No.」でソート（存在しなければソートスキップ）
        if 'No.' in df_cat.columns:
            df_cat = df_cat.sort_values(by='No.').reset_index(drop=True)
        else:
            df_cat = df_cat.reset_index(drop=True)

        # 新しいスライド追加
        slide = prs.slides.add_slide(prs.slide_layouts[5])
        shapes_to_delete = [shape for shape in slide.shapes]
        for shape in shapes_to_delete:
            sp = shape._element
            sp.getparent().remove(sp)

        title_shape = slide.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(9), Inches(0.7))
        title_tf = title_shape.text_frame
        title_tf.text = category
        p = title_tf.paragraphs[0]
        p.font.size = Pt(28)
        p.font.bold = True
        p.alignment = PP_ALIGN.CENTER

        # 「業務カテゴリ」カラム除去
        df_display = df_cat.drop(columns=['業務カテゴリ'])

        rows, cols = df_display.shape
        table_width = Inches(9)
        table_height = Inches(0.3 * rows + 0.8)
        left = Inches(0.5)
        top = Inches(1.0)

        table_shape = slide.shapes.add_table(rows + 1, cols, left, top, table_width, table_height)
        table = table_shape.table

        for col_idx, col_name in enumerate(df_display.columns):
            cell = table.cell(0, col_idx)
            cell.text = str(col_name)
            para = cell.text_frame.paragraphs[0]
            para.font.bold = True
            para.font.size = Pt(12)
            cell.fill.solid()
            cell.fill.fore_color.rgb = RGBColor(200, 200, 200)

        for row_idx in range(rows):
            for col_idx in range(cols):
                cell = table.cell(row_idx + 1, col_idx)
                val = df_display.iat[row_idx, col_idx]
                cell.text = '' if pd.isna(val) else str(val)
                para = cell.text_frame.paragraphs[0]
                para.font.size = Pt(11)
else:
    print("merged_dataが空またはNoneです。PowerPointに追加するデータがありません。")
# 保存


today = datetime.datetime.now()   # datetimeオブジェクト
today = today.strftime("%Y-%m-%d")
pptx_path = "../result/" + str(today) + "_fujiiReport.pptx"
pptx_path = "../報告/" + str(today) + "_fujiiReport.pptx"
prs.save(pptx_path)

print(f"PowerPointファイル '{pptx_path}' を作成しました。")

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 27.33it/s]


PowerPointファイル '../報告/2026-02-12_fujiiReport.pptx' を作成しました。
